# Creative Model: Global LightGBM Cross-Learner

Most time series forecasting methods fit one model per series. This approach does the opposite: it trains a **single global model** across all 100 store-product series at once.

The key idea is that individual series only have ~760 days of data each. A global model pools all 100 series together, giving LightGBM 76,000 training rows to learn from. It can borrow patterns across similar products and stores, which helps especially for shorter or noisier series.

**Why this fits our dataset:**
- We have 100 series of the same type (retail demand), so cross-series learning makes sense.
- We have rich exogenous variables (Promotion, Discount, Competitor Pricing, Weather, Epidemic) that are known ahead of time, which LightGBM can use directly.
- The model is fully automatic: one training run covers all 100 series with no manual tuning per series.

This approach won the M5 Walmart forecasting competition in 2020 and is widely used in industry for large-scale retail forecasting.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import lightgbm as lgb
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error
import warnings
warnings.filterwarnings('ignore')

RANDOM_STATE = 42
TARGET = 'Units Sold'
SHORT_HORIZON = 7
LONG_HORIZON = 30
N_CV_ORIGINS = 15
CV_STEP_DAYS = 7

## 1. Load and Prepare Data

In [ ]:
sales_data = pd.read_csv('sales_data.csv')
sales_data['Date'] = pd.to_datetime(sales_data['Date'])
sales_data = sales_data.sort_values(['Store ID', 'Product ID', 'Date']).reset_index(drop=True)
sales_data['series_id'] = sales_data['Store ID'] + '_' + sales_data['Product ID']

print(f'Total rows: {len(sales_data):,}')
print(f'Date range: {sales_data["Date"].min().date()} to {sales_data["Date"].max().date()}')
print(f'Number of series: {sales_data["series_id"].nunique()}')
print(f'Stores: {sorted(sales_data["Store ID"].unique())}')
print(f'Products: {sales_data["Product ID"].nunique()}')

## 2. Feature Engineering

We create three groups of features:
- **Lag features**: past values of Units Sold (lags 1, 2, 3, 7, 14, 21, 28). These give the model memory of recent demand.
- **Rolling statistics**: rolling means and standard deviations computed from past data. These capture recent demand trends and volatility.
- **Exogenous features**: Promotion, Discount, Competitor Pricing, Weather, Epidemic. These are all known ahead of time (promotions are planned, epidemic status is observable), so they are valid inputs for future forecasts.
- **Time features**: day of week, month, week of year. These help the model learn calendar-based patterns.
- **Static features**: Store ID, Product ID, Category, Region. These let the model distinguish between different series.

In [ ]:
def create_lag_features(df, target_col=TARGET):
    """
    Creates lag and rolling features grouped by series_id.
    All lags are computed with shift(1) minimum so we never leak the current value.
    """
    df = df.copy().sort_values(['series_id', 'Date'])
    grp = df.groupby('series_id')[target_col]

    # Lag features
    for lag in [1, 2, 3, 7, 14, 21, 28]:
        df[f'lag_{lag}'] = grp.shift(lag)

    # Rolling means (shift 1 to avoid leaking current value)
    for window in [7, 14, 28]:
        df[f'rolling_mean_{window}'] = grp.shift(1).transform(
            lambda x: x.rolling(window, min_periods=1).mean()
        )
    
    # Rolling std (volatility signal)
    for window in [7, 14]:
        df[f'rolling_std_{window}'] = grp.shift(1).transform(
            lambda x: x.rolling(window, min_periods=2).std()
        )

    # Time features
    df['day_of_week'] = df['Date'].dt.dayofweek
    df['month'] = df['Date'].dt.month
    df['week_of_year'] = df['Date'].dt.isocalendar().week.astype(int)
    df['day_of_month'] = df['Date'].dt.day
    df['quarter'] = df['Date'].dt.quarter
    df['is_weekend'] = (df['Date'].dt.dayofweek >= 5).astype(int)

    # Encode categorical static features
    cat_cols = ['Store ID', 'Product ID', 'Category', 'Region', 'Weather Condition', 'Seasonality']
    encoders = {}
    for col in cat_cols:
        le = LabelEncoder()
        df[col + '_enc'] = le.fit_transform(df[col].astype(str))
        encoders[col] = le

    return df, encoders


sales_data, encoders = create_lag_features(sales_data)

# Define feature groups
static_features   = ['Store ID_enc', 'Product ID_enc', 'Category_enc', 'Region_enc']
time_features     = ['day_of_week', 'month', 'week_of_year', 'day_of_month', 'quarter', 'is_weekend']
lag_features      = [f'lag_{d}' for d in [1, 2, 3, 7, 14, 21, 28]]
rolling_features  = ['rolling_mean_7', 'rolling_mean_14', 'rolling_mean_28',
                     'rolling_std_7', 'rolling_std_14']
exog_features     = ['Promotion', 'Discount', 'Competitor Pricing', 'Epidemic',
                     'Weather Condition_enc']

ALL_FEATURES = static_features + time_features + lag_features + rolling_features + exog_features

print(f'Total features: {len(ALL_FEATURES)}')
print(f'Feature groups: static={len(static_features)}, time={len(time_features)}, '
      f'lags={len(lag_features)}, rolling={len(rolling_features)}, exog={len(exog_features)}')

## 3. Train / Test Split

We hold out the **last 30 days** as a completely unseen test set. This is a strict temporal holdout: the model never sees any test data during training or hyperparameter tuning. The remaining data is used for training and cross-validation.

In [ ]:
# Drop rows where lag features are NaN (first ~30 rows per series)
df_model = sales_data.dropna(subset=lag_features).reset_index(drop=True)

# Temporal holdout: last 30 days as test set
max_date = df_model['Date'].max()
test_cutoff = max_date - pd.Timedelta(days=LONG_HORIZON)

train_df = df_model[df_model['Date'] <= test_cutoff].copy()
test_df  = df_model[df_model['Date'] >  test_cutoff].copy()

X_train = train_df[ALL_FEATURES]
y_train = train_df[TARGET]
X_test  = test_df[ALL_FEATURES]
y_test  = test_df[TARGET]

print(f'Training rows: {len(train_df):,}   ({train_df["Date"].min().date()} to {train_df["Date"].max().date()})')
print(f'Test rows    : {len(test_df):,}   ({test_df["Date"].min().date()} to {test_df["Date"].max().date()})')
print(f'Training covers all {train_df["series_id"].nunique()} series')

## 4. Train the Global LightGBM Model

We use early stopping on a small validation slice (last 60 days of training) to prevent overfitting. The final model is then trained on all training data.

In [ ]:
LGBM_PARAMS = {
    'objective'        : 'regression_l1',  # MAE loss, more robust to outliers than MSE
    'metric'           : 'mae',
    'learning_rate'    : 0.05,
    'num_leaves'       : 63,
    'min_child_samples': 20,
    'feature_fraction' : 0.8,
    'bagging_fraction' : 0.8,
    'bagging_freq'     : 5,
    'n_estimators'     : 1000,
    'random_state'     : RANDOM_STATE,
    'verbose'          : -1
}

# Use last 60 days of training as an internal validation set for early stopping
val_cutoff = test_cutoff - pd.Timedelta(days=60)
inner_train = train_df[train_df['Date'] <= val_cutoff]
inner_val   = train_df[train_df['Date'] >  val_cutoff]

model_es = lgb.LGBMRegressor(**LGBM_PARAMS)
model_es.fit(
    inner_train[ALL_FEATURES], inner_train[TARGET],
    eval_set=[(inner_val[ALL_FEATURES], inner_val[TARGET])],
    callbacks=[
        lgb.early_stopping(stopping_rounds=50, verbose=False),
        lgb.log_evaluation(period=100)
    ]
)

best_n_estimators = model_es.best_iteration_
print(f'Best number of estimators from early stopping: {best_n_estimators}')

# Retrain on full training set using the best number of estimators
LGBM_PARAMS_FINAL = {**LGBM_PARAMS, 'n_estimators': best_n_estimators}
model = lgb.LGBMRegressor(**LGBM_PARAMS_FINAL)
model.fit(X_train, y_train)
print('Final model trained on full training set.')

## 5. Evaluation Metrics

We use a mix of standard accuracy metrics and business-relevant metrics.

- **MAE / RMSE / MAPE**: standard error measures.
- **Bias**: whether the model systematically over- or under-forecasts.
- **Stockout Rate**: how often the model under-forecasts. In retail, under-forecasting leads to stockouts (lost sales). We want this to be low.
- **Overstock Rate**: how often the model over-forecasts by more than 20%. This leads to excess inventory and holding costs.
- **Directional Accuracy**: how often the model correctly predicts whether demand goes up or down. Useful for ordering decisions.

In [ ]:
def compute_metrics(y_true, y_pred, label='Overall'):
    y_true = np.array(y_true, dtype=float)
    y_pred = np.array(y_pred, dtype=float)

    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))

    # MAPE: skip zero actuals to avoid division by zero
    mask = y_true > 0
    mape = np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100 if mask.sum() > 0 else np.nan

    bias         = np.mean(y_pred - y_true)
    stockout_rate  = np.mean(y_pred < y_true) * 100
    overstock_rate = np.mean(y_pred > y_true * 1.2) * 100

    # Directional accuracy
    if len(y_true) > 1:
        dir_actual = np.sign(np.diff(y_true))
        dir_pred   = np.sign(np.diff(y_pred))
        dir_acc    = np.mean(dir_actual == dir_pred) * 100
    else:
        dir_acc = np.nan

    return {
        'Series'               : label,
        'MAE'                  : round(mae, 2),
        'RMSE'                 : round(rmse, 2),
        'MAPE (%)'             : round(mape, 2),
        'Bias'                 : round(bias, 2),
        'Stockout Rate (%)'    : round(stockout_rate, 2),
        'Overstock Rate (%)'   : round(overstock_rate, 2),
        'Directional Acc. (%)' : round(dir_acc, 2)
    }

## 6. Test Set Results

In [ ]:
# Overall test set performance
test_preds = model.predict(X_test)
overall = compute_metrics(y_test.values, test_preds, label='Global LightGBM')

print('Overall Test Set Metrics (all 100 series, 30-day horizon):')
pd.DataFrame([overall]).set_index('Series')

In [ ]:
# Per-series breakdown
series_metrics = []
for sid in sorted(test_df['series_id'].unique()):
    mask   = test_df['series_id'] == sid
    preds  = model.predict(X_test[mask.values])
    actual = y_test[mask.values].values
    m = compute_metrics(actual, preds, label=sid)
    series_metrics.append(m)

series_df = pd.DataFrame(series_metrics).set_index('Series')

print('Per-series metrics summary (mean across 100 series):')
print(series_df.mean().round(2))
print()
print('Best 5 series by MAE:')
print(series_df.sort_values('MAE').head(5))
print()
print('Worst 5 series by MAE:')
print(series_df.sort_values('MAE', ascending=False).head(5))

## 7. Rolling Origin Cross-Validation

Rolling origin CV gives a more stable estimate of performance than a single holdout split. At each origin, we train on all data up to that point, then evaluate on the next 7 and 30 days. We use 15 origins spaced 7 days apart.

Note: the lag features at validation time are computed from actual observed values. This is the standard academic setup. In a live production system, you would need to forecast lag values recursively for multi-step horizons.

In [ ]:
def rolling_origin_cv(df, feature_cols, target_col, params,
                      n_origins=N_CV_ORIGINS, step_days=CV_STEP_DAYS,
                      h_short=SHORT_HORIZON, h_long=LONG_HORIZON):
    """
    Rolling origin cross-validation for the global LightGBM model.
    Returns a DataFrame with MAE at each origin for both short and long horizons.
    """
    # Keep origins within the training window (before the final test cutoff)
    max_train_date = df['Date'].max() - pd.Timedelta(days=h_long)
    end_origin     = max_train_date - pd.Timedelta(days=h_long)
    start_origin   = end_origin - pd.Timedelta(days=step_days * (n_origins - 1))
    cutoffs = pd.date_range(start=start_origin, end=end_origin, freq=f'{step_days}D')

    results = []
    for cutoff in cutoffs:
        train_cv  = df[df['Date'] <= cutoff].dropna(subset=feature_cols)
        val_short = df[(df['Date'] > cutoff) &
                       (df['Date'] <= cutoff + pd.Timedelta(days=h_short))].dropna(subset=feature_cols)
        val_long  = df[(df['Date'] > cutoff) &
                       (df['Date'] <= cutoff + pd.Timedelta(days=h_long))].dropna(subset=feature_cols)

        if len(train_cv) == 0 or len(val_short) == 0 or len(val_long) == 0:
            continue

        m = lgb.LGBMRegressor(**params)
        m.fit(train_cv[feature_cols], train_cv[target_col])

        mae_short = mean_absolute_error(val_short[target_col], m.predict(val_short[feature_cols]))
        mae_long  = mean_absolute_error(val_long[target_col],  m.predict(val_long[feature_cols]))

        results.append({'Origin': cutoff.date(), 'Horizon': f'{h_short}-day', 'MAE': mae_short})
        results.append({'Origin': cutoff.date(), 'Horizon': f'{h_long}-day',  'MAE': mae_long})
        print(f'  Origin {cutoff.date()}: MAE {h_short}-day = {mae_short:.2f}, MAE {h_long}-day = {mae_long:.2f}')

    return pd.DataFrame(results)


print('Running rolling origin cross-validation...')
cv_results = rolling_origin_cv(df_model, ALL_FEATURES, TARGET, LGBM_PARAMS_FINAL)

print('\nCV Summary (mean MAE per horizon):')
cv_summary = cv_results.groupby('Horizon')['MAE'].agg(['mean', 'std', 'min', 'max']).round(2)
print(cv_summary)

## 8. Visualizations

In [ ]:
# --- Plot 1: Rolling origin CV MAE over time ---
fig, ax = plt.subplots(figsize=(12, 4))
for horizon, grp in cv_results.groupby('Horizon'):
    ax.plot(grp['Origin'].astype(str), grp['MAE'], marker='o', label=horizon)

ax.set_title('Rolling Origin CV: MAE Over Time')
ax.set_xlabel('Origin Date')
ax.set_ylabel('MAE (Units Sold)')
ax.legend(title='Horizon')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# --- Plot 2: Horizon degradation ---
fig, ax = plt.subplots(figsize=(7, 4))
cv_summary_plot = cv_results.groupby('Horizon')['MAE'].mean()
bars = ax.bar(cv_summary_plot.index, cv_summary_plot.values,
              color=['steelblue', 'tomato'], edgecolor='black', linewidth=0.8)
ax.bar_label(bars, fmt='%.1f', padding=3)
ax.set_title('Horizon Degradation: Average MAE by Forecast Horizon')
ax.set_xlabel('Forecast Horizon')
ax.set_ylabel('Mean MAE (Units Sold)')
ax.set_ylim(0, cv_summary_plot.max() * 1.2)
plt.tight_layout()
plt.show()

In [ ]:
# --- Plot 3: Feature importance ---
importance_df = pd.DataFrame({
    'Feature'   : ALL_FEATURES,
    'Importance': model.feature_importances_
}).sort_values('Importance', ascending=False).head(20)

fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(importance_df['Feature'][::-1], importance_df['Importance'][::-1],
        color='steelblue', edgecolor='black', linewidth=0.6)
ax.set_title('Global LightGBM: Top 20 Features by Importance (Split Count)')
ax.set_xlabel('Feature Importance')
plt.tight_layout()
plt.show()

In [ ]:
# --- Plot 4: Forecast vs Actual for 3 sample series (test set) ---
sample_series = ['S001_P0001', 'S003_P0010', 'S005_P0020']
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=False)

for ax, sid in zip(axes, sample_series):
    # Training tail (last 60 days before test)
    train_tail = train_df[train_df['series_id'] == sid].tail(60)
    test_series = test_df[test_df['series_id'] == sid]
    preds = model.predict(test_series[ALL_FEATURES])

    ax.plot(train_tail['Date'], train_tail[TARGET],
            color='steelblue', label='Training (last 60 days)', linewidth=1.2)
    ax.plot(test_series['Date'], test_series[TARGET],
            color='black', label='Actual', linewidth=1.5)
    ax.plot(test_series['Date'], preds,
            color='tomato', linestyle='--', label='Predicted', linewidth=1.5)
    ax.axvline(test_series['Date'].min(), color='grey', linestyle=':', linewidth=1)
    ax.set_title(f'Series: {sid}')
    ax.set_ylabel('Units Sold')
    ax.legend(fontsize=8)

axes[-1].set_xlabel('Date')
fig.suptitle('Global LightGBM: Forecast vs Actual (30-day Test Holdout)', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# --- Plot 5: Per-series MAE distribution ---
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(series_df['MAE'], bins=20, color='steelblue', edgecolor='black', linewidth=0.7)
ax.axvline(series_df['MAE'].mean(), color='tomato', linestyle='--',
           linewidth=1.5, label=f"Mean MAE = {series_df['MAE'].mean():.1f}")
ax.set_title('Distribution of MAE Across 100 Series (30-day Test Holdout)')
ax.set_xlabel('MAE (Units Sold)')
ax.set_ylabel('Number of Series')
ax.legend()
plt.tight_layout()
plt.show()

## 9. Business-Relevant Metric: Asymmetric Cost

Stockouts and overstock are not equally costly in retail. A stockout means a lost sale (revenue loss). Excess inventory means holding costs and potential markdowns. We assign a penalty ratio of 2:1 (under-forecast costs twice as much as over-forecast) and compute a weighted cost metric. This is a custom business metric that goes beyond standard MAE.

In [ ]:
def asymmetric_cost(y_true, y_pred, under_penalty=2.0, over_penalty=1.0):
    """
    Asymmetric cost metric.
    Penalises under-forecasting (stockout risk) more than over-forecasting.
    under_penalty: cost multiplier when prediction < actual
    over_penalty : cost multiplier when prediction > actual
    """
    y_true = np.array(y_true, dtype=float)
    y_pred = np.array(y_pred, dtype=float)
    errors = y_pred - y_true
    cost   = np.where(errors < 0,
                      under_penalty * np.abs(errors),
                      over_penalty  * np.abs(errors))
    return np.mean(cost)


# Naive seasonal benchmark: use the value from 7 days ago as the forecast
naive_preds = test_df.groupby('series_id')[TARGET].shift(7)
# For series where lag_7 is available in the test window, use it
naive_preds = test_df['lag_7'].values  # lag_7 is already computed in df_model

lgbm_cost  = asymmetric_cost(y_test.values, test_preds)
naive_cost = asymmetric_cost(y_test.dropna().values,
                              naive_preds[~np.isnan(naive_preds)])

print(f'Asymmetric Cost (2:1 penalty) -- Global LightGBM : {lgbm_cost:.2f}')
print(f'Asymmetric Cost (2:1 penalty) -- Naive Seasonal  : {naive_cost:.2f}')
print(f'Improvement over naive        : {((naive_cost - lgbm_cost) / naive_cost * 100):.1f}%')

## 10. Summary

The Global LightGBM model:
- Trains a single model across all 100 store-product series, enabling cross-series learning.
- Uses lag features, rolling statistics, time features, and exogenous variables that are known ahead of time (promotions, discounts, epidemic status).
- Produces point forecasts for both short (7-day) and long (30-day) horizons.
- Is fully automatic: no manual intervention is needed per series.

**Key limitation:** For the 30-day horizon, weather and epidemic status may not be fully known in advance. In that case, average historical values per month can be used as a proxy, which slightly reduces accuracy but keeps the model practically deployable.

**Comparison with simple model (Holt-Winters) and max performance model (SARIMA ensemble):** See the final results table in the Validation section of the report.